# 04 · Evaluación

Reglas de decisión (argmax, umbral binario, mínimo costo esperado), costo total en test y bootstrap pareado al 95 %.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append('..')
import numpy as np, pandas as pd
import utils

In [ ]:
rows = []
for form in ['convencional', 'landmarking']:
    cal = utils.load_predictions('lgbm', form, 'cal')
    val = utils.load_predictions('lgbm', form, 'validation')
    Pc, Pv, yv = cal[utils.PRED_COLS].to_numpy(), val[utils.PRED_COLS].to_numpy(), val['y'].to_numpy()

    calibrators = {'sin calibrar': None,
                   'temperatura': utils.TemperatureScaler(),
                   'isotónica': utils.IsotonicOVR()}
    for cname, C in calibrators.items():
        Q = Pv if C is None else C.fit(Pc, cal['y'], cal['w']).transform(Pv)
        theta, _ = utils.best_threshold(Q, yv)
        for rule, yhat in [('argmax', utils.decide_argmax(Q)),
                           (f'umbral (θ={theta:.3f})', utils.decide_threshold(Q, theta)),
                           ('min_costo', utils.decide_min_cost(Q))]:
            rows.append({'formulación': form, 'calibración': cname, 'regla': rule,
                         'Brier': round(utils.brier(yv, Q), 4),
                         'costo_total': utils.total_cost(yv, yhat),
                         '% a revisión': round(100 * (yhat > 0).mean(), 1)})

tab = pd.DataFrame(rows)
tab.to_csv(utils.RESULTS / 'lgbm_calibracion_reglas_validation.csv', index=False)
tab

In [ ]:
from sklearn.model_selection import StratifiedKFold

class PriorShift:
    """Reescala cada clase por π_objetivo / π_origen y renormaliza."""
    def fit(self, P, y, w=None):
        pi_target = np.bincount(np.asarray(y, int), minlength=5) + 1.0
        self.r_ = (pi_target / pi_target.sum()) / P.mean(0)
        return self
    def transform(self, P):
        Q = P * self.r_
        return Q / Q.sum(1, keepdims=True)

methods = {'sin calibrar': None,
           'temperatura': utils.TemperatureScaler,
           'isotónica':   utils.IsotonicOVR,
           'prior':       PriorShift}

def crossfit_eval(Pv, yv, method, k=5, seed=0):
    """Calibración y umbral ajustados en k-1 grupos de validation, evaluados en el restante."""
    yhat = {r: np.zeros(len(yv), int) for r in ['argmax', 'umbral', 'min_costo']}
    Q = np.zeros_like(Pv)
    for tr, te in StratifiedKFold(k, shuffle=True, random_state=seed).split(Pv, yv):
        C = None if method is None else method().fit(Pv[tr], yv[tr])
        Qtr = Pv[tr] if C is None else C.transform(Pv[tr])
        Qte = Pv[te] if C is None else C.transform(Pv[te])
        theta, _ = utils.best_threshold(Qtr, yv[tr])
        yhat['argmax'][te]    = utils.decide_argmax(Qte)
        yhat['umbral'][te]    = utils.decide_threshold(Qte, theta)
        yhat['min_costo'][te] = utils.decide_min_cost(Qte)
        Q[te] = Qte
    return Q, yhat

rows = []
for form in ['convencional', 'landmarking']:
    val = utils.load_predictions('lgbm', form, 'validation')
    Pv, yv = val[utils.PRED_COLS].to_numpy(), val['y'].to_numpy()
    for mname, M in methods.items():
        Q, yhat = crossfit_eval(Pv, yv, M)
        for rule, yh in yhat.items():
            rows.append({'formulación': form, 'calibración': mname, 'regla': rule,
                         'Brier': round(utils.brier(yv, Q), 4),
                         'costo_total': utils.total_cost(yv, yh),
                         '% a revisión': round(100 * (yh > 0).mean(), 1)})

tab_cf = pd.DataFrame(rows)
tab_cf.to_csv(utils.RESULTS / 'lgbm_reglas_validacion_cruzada.csv', index=False)
tab_cf.pivot_table(index=['formulación', 'calibración'], columns='regla',
                   values='costo_total').round(0)

In [ ]:
# Brier por calibración (el diseño elige el método con menor Brier)
print(tab_cf.groupby(['formulación', 'calibración'])['Brier'].first().unstack().round(4), '\n')

preds_cf = {}
for form in ['convencional', 'landmarking']:
    val = utils.load_predictions('lgbm', form, 'validation')
    Pv, yv = val[utils.PRED_COLS].to_numpy(), val['y'].to_numpy()
    for mname, M in methods.items():
        preds_cf[(form, mname)] = crossfit_eval(Pv, yv, M)[1]

comparaciones = [
    # (descripción, (form, calib, regla) A, (form, calib, regla) B)
    ('H2 conv: min_costo vs umbral',  ('convencional', 'temperatura', 'min_costo'), ('convencional', 'temperatura', 'umbral')),
    ('H2 lm: min_costo vs umbral',    ('landmarking', 'isotónica', 'min_costo'),    ('landmarking', 'isotónica', 'umbral')),
    ('H2 lm: min_costo vs argmax',    ('landmarking', 'isotónica', 'min_costo'),    ('landmarking', 'isotónica', 'argmax')),
    ('H1: landmarking vs conv (umbral)', ('landmarking', 'isotónica', 'umbral'),    ('convencional', 'temperatura', 'umbral')),
    ('Mejor vs siempre 4',            ('landmarking', 'isotónica', 'umbral'),       None),
]

rows = []
for desc, a, b in comparaciones:
    ya = preds_cf[(a[0], a[1])][a[2]]
    yb = np.full(len(yv), 4) if b is None else preds_cf[(b[0], b[1])][b[2]]
    rows.append({'comparación': desc, **utils.paired_bootstrap(yv, ya, yb)})

boot = pd.DataFrame(rows)
boot.to_csv(utils.RESULTS / 'lgbm_bootstrap_validation.csv', index=False)
boot